# LLM Coding and Human–LLM Agreement (Notebook 03)

This notebook runs the **LLM auto-coder** over the 88-episode feasibility sample
and, once a human-coded copy exists, reports **human–LLM agreement**.

**Design (decided):** the human annotator's labels are the **gold standard**. The
LLM's labels play two honest roles: (a) the Section-5 zero/few-shot **benchmark
baseline**, scored against the human gold, and (b) a reported human–LLM
**agreement diagnostic**. The diagnostic is *not* the reliability coefficient —
Krippendorff's α as *reliability* is reserved for human–human coding. This avoids
the circularity of an LLM grading LLM-made ground truth.

**Data governance.** The default backend is an **offline mock** so the pipeline
runs with no external calls. The real backend (`anthropic`) sends episode text
(Polish + machine-English + context) to the Anthropic API. That is an external
service: enable it only after confirming the XAI-FUNGI licence and original
consent permit it, by setting `ALLOW_EXTERNAL_LLM = True`. The run log records the
exact model, date, and parameters for the ACM/IUI methods disclosure.

Both coders use `docs/contestation_coding_guide.md` verbatim.

In [1]:
# ============================================================
# 1. Config, paths, taxonomy vocabulary
# ============================================================

from __future__ import annotations

import hashlib
import json
import logging
import platform
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from IPython.display import display

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
LOGGER = logging.getLogger("xai_contestation")


def find_project_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root (a parent with data/).")


PROJECT_ROOT = find_project_root()
AUDIT_DIR = PROJECT_ROOT / "outputs" / "02_translation_and_feasibility_audit" / "tables"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "03_llm_coding_and_agreement"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CODING_GUIDE = PROJECT_ROOT / "docs" / "contestation_coding_guide.md"
REVIEW_SAMPLE = AUDIT_DIR / "feasibility_review_sample.csv"
HUMAN_CODED = AUDIT_DIR / "feasibility_review_sample_HUMAN.csv"   # you produce this
LLM_CODED = OUTPUT_DIR / "feasibility_review_sample_LLM.csv"
LLM_CACHE = OUTPUT_DIR / "llm_cache.json"
RUN_LOG = OUTPUT_DIR / "llm_run_log.json"

# ------------------------------------------------------------
# LLM backend configuration
# ------------------------------------------------------------
BACKEND = "mock"                 # "mock" (offline default) or "anthropic"
ALLOW_EXTERNAL_LLM = False        # set True only after licence/consent review
LLM_MODEL = "claude-opus-5"       # disclosed in the run log; swap per resources
LLM_EFFORT = "medium"             # low | medium | high | xhigh | max
LLM_MAX_TOKENS = 2048

# ------------------------------------------------------------
# Controlled vocabularies (must match docs/contestation_coding_guide.md)
# ------------------------------------------------------------
VOCAB = {
    "is_contestation": ["yes", "no"],
    "presence": ["explicit", "implicit", "ambiguous", "absent"],
    "target": ["prediction", "reasoning", "evidence", "input data",
               "explanation representation", "system competence", "n/a"],
    "interaction_act": ["request for justification", "direct challenge",
                        "correction", "alternative proposal", "counterexample",
                        "rejection", "request to acknowledge uncertainty", "n/a"],
    "grounds": ["domain knowledge", "technical knowledge", "prior experience",
                "internal inconsistency", "missing evidence", "data-quality concern",
                "causal implausibility", "presentation ambiguity", "n/a"],
    "expected_response": ["justify", "revise", "incorporate correction",
                          "request additional information", "acknowledge uncertainty",
                          "expose conflicting evidence", "defer to human expertise", "n/a"],
}
DIMENSIONS = list(VOCAB)

LOGGER.info("Project root: %s", PROJECT_ROOT)
LOGGER.info("Backend: %s (external calls allowed: %s)", BACKEND, ALLOW_EXTERNAL_LLM)

INFO | Project root: /Users/sabrimanai/software/uj/detecting-contestation-xai
INFO | Backend: mock (external calls allowed: False)


In [2]:
# ============================================================
# 2. Load the review sample and the coding guide
# ============================================================

if not REVIEW_SAMPLE.exists():
    raise FileNotFoundError(
        f"{REVIEW_SAMPLE} not found. Run notebook 02 first to generate the sample.")

sample = pd.read_csv(REVIEW_SAMPLE)
LOGGER.info("Loaded %d review episodes.", len(sample))

if CODING_GUIDE.exists():
    CODEBOOK_TEXT = CODING_GUIDE.read_text(encoding="utf-8")
else:  # minimal fallback so the notebook still runs
    CODEBOOK_TEXT = (
        "Contestation = an explicit or implicit challenge to an AI system's "
        "conclusion, reasoning, evidence, explanation, or epistemic authority. "
        "Distinguish it from incomprehension, expressed uncertainty, and aesthetic "
        "criticism. Code from the Polish text.")
    LOGGER.warning("Coding guide not found at %s; using fallback definition.", CODING_GUIDE)

display(sample[["__utt_id", "participant_group", "explanation_format",
                "text_pl", "text_en"]].head(4))

INFO | Loaded 88 review episodes.


,__utt_id,participant_group,explanation_format,text_pl,text_en
0,DR_SSH_07:459,SSH,LIME,"szersze, wyższe, a nie takie jak kulka.","wider, taller, not like a little ball."
1,PK_DE_01:207,DE,Anchor,"Nieodpowiedni termin tu jest, tak? Według pani...","The term here is wrong, isn't it? In your know..."
2,PK_DE_05:105,DE,SHAP,"Znaczy tak: same te wartości wszystkie, które ...",So: all these values that equal zero — the que...
3,PK_DE_04:162,DE,LIME,"Tak!! i tu jest, to jest tutaj całkowicie właś...","Yes!! and here it is, this is exactly it... Th..."


In [3]:
# ============================================================
# 3. Prompt + structured-output contract
# ============================================================
#
# The Anthropic backend constrains output with a JSON schema built directly
# from VOCAB (each dimension is an enum), so there is no separate model class
# to keep in sync.

SYSTEM_PROMPT = (
    CODEBOOK_TEXT
    + "\n\n---\nYou are the LLM auto-coder. Apply the codebook above to ONE "
      "episode. Code from the Polish `text_pl` in its `context_pl`; the English is "
      "a machine-translation aid only. Return exactly these fields with values drawn "
      "verbatim from the controlled vocabularies; when is_contestation is 'no', set "
      "the four structure fields (target, interaction_act, grounds, expected_response) "
      "to 'n/a' and presence to 'absent'.\n"
    + "Vocabularies:\n"
    + "\n".join(f"- {dim}: {sorted(vals)}" for dim, vals in VOCAB.items())
)


def build_user_prompt(row: pd.Series) -> str:
    return (
        f"Episode id: {row['__utt_id']}\n"
        f"Participant group: {row['participant_group']}\n"
        f"Explanation format: {row['explanation_format']}\n\n"
        f"text_pl: {row['text_pl']}\n"
        f"context_pl: {row['context_pl']}\n\n"
        f"text_en (aid): {row.get('text_en')}\n"
        f"context_en (aid): {row.get('context_en')}\n"
    )


def coerce_label(dim: str, value) -> object:
    """Snap a returned value into the controlled vocabulary; flag invalids."""
    if value is None:
        return "n/a" if dim != "is_contestation" else pd.NA
    v = str(value).strip().lower()
    allowed = {a.lower(): a for a in VOCAB[dim]}
    return allowed.get(v, f"INVALID:{value}")


PROMPT_FINGERPRINT = hashlib.sha1(SYSTEM_PROMPT.encode("utf-8")).hexdigest()[:12]
LOGGER.info("Prompt fingerprint: %s", PROMPT_FINGERPRINT)

INFO | Prompt fingerprint: 4973b1a6db6a


In [4]:
# ============================================================
# 4. Backends: offline mock (default) and Anthropic (real)
# ============================================================

def mock_backend(row: pd.Series) -> dict:
    """Deterministic heuristic coder for pipeline testing ONLY. Not for reporting.

    Uses the lexical hit columns from notebook 02 as a stand-in signal.
    """
    challenge = int(row.get("challenge_hits", 0) or 0)
    incomp = int(row.get("incomprehension_hits", 0) or 0)
    is_c = "yes" if challenge >= 2 and challenge > incomp else "no"
    if is_c == "no":
        return {"is_contestation": "no", "presence": "absent", "target": "n/a",
                "interaction_act": "n/a", "grounds": "n/a",
                "expected_response": "n/a", "notes": "mock"}
    return {"is_contestation": "yes", "presence": "explicit", "target": "prediction",
            "interaction_act": "direct challenge", "grounds": "domain knowledge",
            "expected_response": "justify", "notes": "mock"}


def build_anthropic_backend() -> Callable[[pd.Series], dict]:
    """Return a per-episode coder backed by the Anthropic API (structured output)."""
    if not ALLOW_EXTERNAL_LLM:
        raise RuntimeError(
            "External LLM disabled. Set ALLOW_EXTERNAL_LLM=True only after confirming "
            "the dataset licence and consent permit sending episode text to the API.")
    import anthropic

    client = anthropic.Anthropic()   # resolves creds from env / `ant auth login`
    LOGGER.info("Anthropic backend ready (model=%s, effort=%s).", LLM_MODEL, LLM_EFFORT)

    def code_one(row: pd.Series) -> dict:
        # No temperature/top_p: removed on current models (would 400).
        resp = client.messages.create(
            model=LLM_MODEL,
            max_tokens=LLM_MAX_TOKENS,
            output_config={
                "effort": LLM_EFFORT,
                "format": {
                    "type": "json_schema",
                    "schema": {
                        "type": "object",
                        "properties": {
                            dim: {"type": "string", "enum": VOCAB[dim]}
                            for dim in DIMENSIONS
                        } | {"notes": {"type": "string"}},
                        "required": DIMENSIONS + ["notes"],
                        "additionalProperties": False,
                    },
                },
            },
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": build_user_prompt(row)}],
        )
        code_one.served_model = resp.model      # for the disclosure log
        text = next(b.text for b in resp.content if b.type == "text")
        return json.loads(text)

    code_one.served_model = None
    return code_one


def build_backend() -> Callable[[pd.Series], dict]:
    if BACKEND == "anthropic":
        return build_anthropic_backend()
    LOGGER.warning("Using OFFLINE MOCK backend — outputs are NOT valid annotations.")
    return mock_backend

In [5]:
# ============================================================
# 5. Run the LLM coder over the sample (cached, idempotent)
# ============================================================

def load_cache() -> dict:
    return json.loads(LLM_CACHE.read_text()) if LLM_CACHE.exists() else {}


def save_cache(cache: dict) -> None:
    LLM_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1))


def run_llm_coder(df: pd.DataFrame) -> pd.DataFrame:
    backend = build_backend()
    cache = load_cache()
    cache_key = lambda uid: f"{BACKEND}:{LLM_MODEL}:{PROMPT_FINGERPRINT}:{uid}"
    served_model = None
    rows = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        key = cache_key(row["__utt_id"])
        if key not in cache:
            cache[key] = backend(row)
            if i % 10 == 0:
                save_cache(cache)
                LOGGER.info("Coded %d / %d ...", i, len(df))
        served_model = getattr(backend, "served_model", None) or served_model
        coded = {dim: coerce_label(dim, cache[key].get(dim)) for dim in DIMENSIONS}
        coded["notes"] = cache[key].get("notes", "")
        coded["__utt_id"] = row["__utt_id"]
        rows.append(coded)
    save_cache(cache)

    # The review sample carries BLANK annotator columns (is_contestation, the five
    # dimensions, annotator, notes). Drop them so the LLM labels take the bare
    # column names; cell 7 then diffs those against the human-coded file.
    placeholder = DIMENSIONS + ["annotator", "notes"]
    base = df.drop(columns=[c for c in placeholder if c in df.columns])
    result = base.merge(pd.DataFrame(rows), on="__utt_id")

    # Disclosure log for the ACM/IUI methods section.
    RUN_LOG.write_text(json.dumps({
        "backend": BACKEND,
        "model_requested": LLM_MODEL,
        "model_served": served_model,
        "effort": LLM_EFFORT,
        "max_tokens": LLM_MAX_TOKENS,
        "temperature": "unset (removed on current models)",
        "prompt_fingerprint": PROMPT_FINGERPRINT,
        "n_episodes": int(len(df)),
        "date": date.today().isoformat(),
        "anthropic_sdk": (__import__("anthropic").__version__
                          if BACKEND == "anthropic" else None),
        "python": platform.python_version(),
    }, indent=2))
    return result


llm_coded = run_llm_coder(sample)
llm_cols = ["__utt_id", "participant_group", "explanation_format",
            "is_contestation", "presence", "target", "interaction_act",
            "grounds", "expected_response", "notes"]
llm_coded[llm_cols].to_csv(LLM_CODED, index=False)
LOGGER.info("Wrote LLM codings to %s", LLM_CODED)
display(llm_coded[llm_cols].head(8))
print("is_contestation (LLM):")
print(llm_coded["is_contestation"].value_counts(dropna=False).to_string())

WARNING | Using OFFLINE MOCK backend — outputs are NOT valid annotations.
INFO | Coded 10 / 88 ...
INFO | Coded 20 / 88 ...
INFO | Coded 30 / 88 ...
INFO | Coded 40 / 88 ...
INFO | Coded 50 / 88 ...
INFO | Coded 60 / 88 ...
INFO | Coded 70 / 88 ...
INFO | Coded 80 / 88 ...
INFO | Wrote LLM codings to /Users/sabrimanai/software/uj/detecting-contestation-xai/outputs/03_llm_coding_and_agreement/feasibility_review_sample_LLM.csv


,__utt_id,participant_group,explanation_format,is_contestation,presence,target,interaction_act,grounds,expected_response,notes
0,DR_SSH_07:459,SSH,LIME,no,absent,n/a,n/a,n/a,n/a,mock
1,PK_DE_01:207,DE,Anchor,yes,explicit,prediction,direct challenge,domain knowledge,justify,mock
2,PK_DE_05:105,DE,SHAP,yes,explicit,prediction,direct challenge,domain knowledge,justify,mock
3,PK_DE_04:162,DE,LIME,no,absent,n/a,n/a,n/a,n/a,mock
4,PK_DE_01:190,DE,Descriptive statistics,yes,explicit,prediction,direct challenge,domain knowledge,justify,mock
5,MZ_SSH_04:169,SSH,Counterfactual,no,absent,n/a,n/a,n/a,n/a,mock
6,PK_DE_03:64,DE,Counterfactual,no,absent,n/a,n/a,n/a,n/a,mock
7,PK_DE_01:121,DE,LIME,no,absent,n/a,n/a,n/a,n/a,mock


is_contestation (LLM):
is_contestation
no     72
yes    16


In [6]:
# ============================================================
# 6. Nominal Krippendorff's alpha (2 coders) — diagnostic use
# ============================================================

def krippendorff_alpha_nominal(pairs: list[tuple]) -> float:
    """Nominal Krippendorff's alpha for units rated by exactly two coders.

    `pairs` is a list of (value_a, value_b) for units both coders labelled
    (units with a missing label are dropped by the caller).
    """
    o = Counter()
    n = 0
    for a, b in pairs:
        o[(a, b)] += 1
        o[(b, a)] += 1
        n += 2
    if n == 0:
        return float("nan")
    marg = Counter()
    for (v, _vp), c in o.items():
        marg[v] += c
    do = sum(c for (v, vp), c in o.items() if v != vp) / n
    vals = list(marg)
    de = sum(marg[v] * marg[vp] for v in vals for vp in vals if v != vp) / (n * (n - 1))
    if de == 0:
        return 1.0
    return 1.0 - do / de


# self-test on trivial cases
assert abs(krippendorff_alpha_nominal([("a", "a"), ("b", "b"), ("c", "c")]) - 1.0) < 1e-9
assert krippendorff_alpha_nominal([("a", "b"), ("b", "a")]) < 0.0
LOGGER.info("Krippendorff alpha self-test passed.")

INFO | Krippendorff alpha self-test passed.


In [7]:
# ============================================================
# 7. Human–LLM agreement + gate read (runs once human coding exists)
# ============================================================

if not HUMAN_CODED.exists():
    print("No human-coded file yet at:\n  ", HUMAN_CODED)
    print("\nTo produce it: code feasibility_review_sample.csv by hand (blind to the")
    print("LLM), filling is_contestation + the five taxonomy columns, and save it under")
    print("that name. Re-run this cell to get agreement, the lexical-filter precision,")
    print("and the gate numbers for main-2.tex.")
else:
    human = pd.read_csv(HUMAN_CODED)
    merged = human.merge(llm_coded, on="__utt_id", suffixes=("_h", "_llm"))

    # Per-dimension human-LLM agreement (raw + nominal alpha, DIAGNOSTIC only).
    agree_rows = []
    for dim in DIMENSIONS:
        h, m = merged[f"{dim}_h"].astype("string"), merged[f"{dim}_llm"].astype("string")
        mask = h.notna() & m.notna()
        pairs = list(zip(h[mask], m[mask]))
        raw = float(np.mean([a == b for a, b in pairs])) if pairs else float("nan")
        agree_rows.append({"dimension": dim, "n": int(mask.sum()),
                           "raw_agreement": round(raw, 3),
                           "alpha_diagnostic": round(krippendorff_alpha_nominal(pairs), 3)})
    agreement = pd.DataFrame(agree_rows)
    print("Human–LLM agreement (DIAGNOSTIC — not the reliability coefficient):")
    display(agreement)
    agreement.to_csv(OUTPUT_DIR / "human_llm_agreement.csv", index=False)

    # is_contestation confusion (human gold rows, LLM columns).
    print("\nis_contestation confusion (rows=human gold, cols=LLM):")
    display(pd.crosstab(merged["is_contestation_h"], merged["is_contestation_llm"]))

    # Lexical-filter precision from the human gold.
    pos = (merged["is_contestation_h"].astype("string") == "yes").sum()
    precision = pos / len(merged) if len(merged) else float("nan")
    print(f"\nLexical-filter precision (human gold): {pos}/{len(merged)} = {precision:.0%}")

    # Diversity of confirmed contestation across the 5-format x 3-group grid.
    conf = merged[merged["is_contestation_h"].astype("string") == "yes"]
    print("\nConfirmed contestation episodes by group x format:")
    display(pd.crosstab(conf["participant_group_h"] if "participant_group_h" in conf
                        else conf["participant_group"],
                        conf["explanation_format_h"] if "explanation_format_h" in conf
                        else conf["explanation_format"]))

    # Adjudication sheet: rows where human and LLM disagree on presence.
    disagree = merged[merged["is_contestation_h"].astype("string")
                      != merged["is_contestation_llm"].astype("string")]
    disagree.to_csv(OUTPUT_DIR / "human_llm_disagreements.csv", index=False)
    print(f"\n{len(disagree)} presence disagreements saved for adjudication.")

    print("\n>> GATE: report confirmed-episode count, group/format diversity, and the")
    print("   contestation-vs-epistemic-friction verdict into main-2.tex Sec. 3 (sec:audit).")

Human–LLM agreement (DIAGNOSTIC — not the reliability coefficient):


,dimension,n,raw_agreement,alpha_diagnostic
0,is_contestation,88,0.534,-0.008
1,presence,88,0.432,-0.044
2,target,47,0.000,-0.275
3,interaction_act,47,0.043,-0.228
4,grounds,47,0.064,-0.202
5,expected_response,47,0.043,-0.258



is_contestation confusion (rows=human gold, cols=LLM):


is_contestation_llm,no,yes
is_contestation_h,,
no,36,5
yes,36,11



Lexical-filter precision (human gold): 47/88 = 53%

Confirmed contestation episodes by group x format:


explanation_format_h,Anchor,Counterfactual,Descriptive statistics,LIME,SHAP
participant_group_h,,,,,
DE,4,4,4,3,2
IT,1,3,5,2,2
SSH,3,4,4,1,5



41 presence disagreements saved for adjudication.

>> GATE: report confirmed-episode count, group/format diversity, and the
   contestation-vs-epistemic-friction verdict into main-2.tex Sec. 3 (sec:audit).


## How to use this notebook

1. **Human pass first.** Hand-code `feasibility_review_sample.csv` (blind to the
   LLM), save as `feasibility_review_sample_HUMAN.csv` in the same folder.
2. **LLM pass.** To use the real model, set `BACKEND = "anthropic"` and
   `ALLOW_EXTERNAL_LLM = True` (after the licence/consent check), ensure
   `pip install anthropic` and credentials (`ANTHROPIC_API_KEY` or `ant auth login`),
   then run cell 5. The offline mock runs with no setup but its output is **not** a
   valid annotation — it only exercises the pipeline.
3. **Agreement + gate.** Re-run cell 7 to get human–LLM agreement (diagnostic),
   the confusion matrix, the lexical-filter precision, the confirmed-episode
   diversity grid, and the adjudication sheet.
4. **Disclosure.** `llm_run_log.json` records the model, served version, date, and
   parameters — paste these into the methods (`sec:models`), per ACM/IUI policy.
5. **Reliability caveat.** Report human–LLM agreement as a diagnostic only. For
   Krippendorff's α *as reliability*, add a second human coder on a subset.